# 5. Can we make the model fairer?

**Problem found in notebook 3:** the model finds 76% of sick men but only 57% of sick women.
**Reason found in notebook 4:** the model learned that being a woman means lower risk.

Here we try three simple changes and see what happens:

1. **Remove sex** from the model.
2. **Lower the decision threshold** for everyone (call a patient "sick" from risk 0.4 instead of 0.5).
3. **Lower the threshold only for women.**

For each version we look at sensitivity (sick patients found) and specificity (healthy patients correctly called healthy), for women and men.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

data = pd.read_csv("../data/clean.csv")
y = data["target"]
is_woman = data["sex"] == 0

print("Sick women:", (y[is_woman] == 1).sum(), " Sick men:", (y[~is_woman] == 1).sum())

## Step 1: get the risk for every patient
Same model and same cross-validation as notebook 2 (each patient predicted by a model that never saw them).
The function lets us choose **which features** the model uses.

In [ ]:
def cross_validated_risk(continuous, categorical):
    prepare = ColumnTransformer([
        ("continuous", StandardScaler(), continuous),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    model = Pipeline([
        ("prepare", prepare),
        ("classifier", LogisticRegression(max_iter=1000)),
    ])
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X = data[continuous + categorical]
    return cross_val_predict(model, X, y, cv=folds, method="predict_proba")[:, 1]


continuous = ["age", "trestbps", "chol", "thalach", "oldpeak"]
categorical_with_sex = ["sex", "cp", "fbs", "restecg", "exang"]
categorical_without_sex = ["cp", "fbs", "restecg", "exang"]

risk_with_sex = cross_validated_risk(continuous, categorical_with_sex)
risk_without_sex = cross_validated_risk(continuous, categorical_without_sex)
print("Done")

## Step 2: a function that measures one version
- `predicted`: 1 (sick) or 0 (healthy) for each patient, after applying the threshold.
- **Sensitivity** = sick patients found / all sick patients
- **Specificity** = healthy patients called healthy / all healthy patients

In [ ]:
def measure(name, predicted):
    row = {"version": name}
    for group, mask in [("women", is_woman), ("men", ~is_woman)]:
        sick = (y == 1) & mask
        healthy = (y == 0) & mask
        row["sens_" + group] = (predicted[sick] == 1).mean()
        row["spec_" + group] = (predicted[healthy] == 0).mean()
    row["sick_women_found"] = f"{(predicted[(y == 1) & is_woman] == 1).sum()} / {((y == 1) & is_woman).sum()}"
    row["accuracy"] = (predicted == y).mean()
    return row

## Step 3: compare the versions
- **A. Original:** with sex, threshold 0.5 (same as notebook 2)
- **B. Without sex:** threshold 0.5
- **C. Lower threshold for everyone:** with sex, threshold 0.4
- **D. Lower threshold for women only:** with sex, 0.35 for women, 0.5 for men

In [ ]:
threshold_D = np.where(is_woman, 0.35, 0.5)

results = pd.DataFrame([
    measure("A. Original",               (risk_with_sex >= 0.5).astype(int)),
    measure("B. Without sex",            (risk_without_sex >= 0.5).astype(int)),
    measure("C. Threshold 0.4 for all",  (risk_with_sex >= 0.4).astype(int)),
    measure("D. Threshold 0.35 women",   (risk_with_sex >= threshold_D).astype(int)),
]).round(3)

results

## Step 4: sensitivity for women and men, by version

In [ ]:
x = np.arange(len(results))
plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, results["sens_women"], width=0.4, label="Women")
plt.bar(x + 0.2, results["sens_men"], width=0.4, label="Men")
plt.xticks(x, results["version"], rotation=15)
plt.ylabel("Sensitivity (sick patients found)")
plt.ylim(0, 1)
plt.legend()
plt.title("Sick patients found, by version")
plt.tight_layout()
plt.show()

## What to look for
- Does the **gap** between women and men get smaller?
- What is the **price**? Finding more sick patients usually means more false alarms (lower specificity).
- **Version B** is not automatically fair: other features can carry information about sex.
- **Version D** treats women and men with different rules. It can reduce the gap, but using different thresholds by sex is debated, and the reason must be stated openly.
- Only 37 sick women: differences of a few patients change the numbers a lot.